In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [13]:
df = pd.read_csv('/home1/bingi' + '/mimic-iv-statstest.csv')
test = pd.read_csv('/home1/bingi' + '/mimic-iii-statstest.csv')

df.head()

,Unnamed: 0,first_careunit,last_careunit,los,age,is_stroke,insurance,ethnicity,pvd,htn,...,max_systolic_BP,max_diastolic_BP,max_mean_BP,max_Spo2,min_Heart Rate,min_systolic_BP,min_diastolic_BP,min_mean_BP,min_Spo2,gender
0,0,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),3.427512,59,0,Other,WHITE,1.0,0.0,...,121.0,61.0,76.0,100.0,57.0,100.0,56.0,65.0,95.0,F
1,1,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),3.871100,76,1,Medicare,WHITE,0.0,1.0,...,152.0,72.0,90.0,98.0,60.0,123.0,50.0,66.0,94.0,M
2,2,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2.982535,68,0,Medicare,WHITE,0.0,1.0,...,141.0,85.0,88.0,99.0,61.0,83.0,34.0,50.0,94.0,F
3,3,Surgical Intensive Care Unit (SICU),Trauma SICU (TSICU),1.144005,63,0,Other,WHITE,1.0,1.0,...,NaN,NaN,NaN,100.0,76.0,NaN,NaN,NaN,92.0,F
4,4,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),3.891053,63,1,Other,WHITE,0.0,1.0,...,146.0,94.0,99.0,100.0,93.0,82.0,49.0,65.0,93.0,F


In [14]:
df = df.drop(columns=['Unnamed: 0'], errors = 'ignore')
#df = df.set_index('key_0')

df.columns

Index(['first_careunit', 'last_careunit', 'los', 'age', 'is_stroke',
       'insurance', 'ethnicity', 'pvd', 'htn', 'copd', 'diabetes', 'renal',
       'liver', 'pud', 'sepsis', 'chf', 'cancer', 'rheumatic', 'Alcohol',
       'Tobacco', 'Opioid', 'Cannabis', 'Cocaine', 'Sedative', 'Other',
       'Smoking', 'mean_Anion Gap', 'mean_Ca', 'mean_chloride',
       'mean_Creatinine', 'mean_Glucose', 'mean_Mg', 'mean_Phosphate',
       'mean_Potassium', 'mean_Na', 'mean_Urea Nitrogen', 'mean_INR(PT)',
       'mean_Platelet Count', 'mean_PT', 'mean_PTT', 'mean_WBC',
       'max_Anion Gap', 'max_Ca', 'max_chloride', 'max_Creatinine',
       'max_Glucose', 'max_Mg', 'max_Phosphate', 'max_Potassium', 'max_Na',
       'max_Urea Nitrogen', 'max_INR(PT)', 'max_Platelet Count', 'max_PT',
       'max_PTT', 'max_WBC', 'min_Anion Gap', 'min_Ca', 'min_chloride',
       'min_Creatinine', 'min_Glucose', 'min_Mg', 'min_Phosphate',
       'min_Potassium', 'min_Na', 'min_Urea Nitrogen', 'min_INR(PT)',
       

In [15]:
test = test.drop(columns=['Unnamed: 0'], errors = 'ignore')
#test = test.set_index('key_0')

test.columns

Index(['first_careunit', 'last_careunit', 'los', 'age', 'is_stroke',
       'insurance', 'ethnicity', 'pvd', 'htn', 'copd', 'diabetes', 'renal',
       'liver', 'pud', 'sepsis', 'chf', 'cancer', 'rheumatic', 'Alcohol',
       'Tobacco', 'Opioid', 'Cannabis', 'Cocaine', 'Sedative', 'Other',
       'Smoking', 'mean_Anion Gap', 'mean_Ca', 'mean_chloride',
       'mean_Creatinine', 'mean_Glucose', 'mean_Mg', 'mean_Phosphate',
       'mean_Potassium', 'mean_Na', 'mean_Urea Nitrogen', 'mean_INR(PT)',
       'mean_Platelet Count', 'mean_PT', 'mean_PTT', 'mean_WBC',
       'min_Anion Gap', 'min_Ca', 'min_chloride', 'min_Creatinine',
       'min_Glucose', 'min_Mg', 'min_Phosphate', 'min_Potassium', 'min_Na',
       'min_Urea Nitrogen', 'min_INR(PT)', 'min_Platelet Count', 'min_PT',
       'min_PTT', 'min_WBC', 'max_Anion Gap', 'max_Ca', 'max_chloride',
       'max_Creatinine', 'max_Glucose', 'max_Mg', 'max_Phosphate',
       'max_Potassium', 'max_Na', 'max_Urea Nitrogen', 'max_INR(PT)',
       

In [16]:
num_cols = df.select_dtypes(include=['float64','int64']).columns
cat_cols = df.select_dtypes(include=['object','category']).columns

test_num_cols = test.select_dtypes(include=['float64','int64']).columns
test_cat_cols = test.select_dtypes(include=['object','category']).columns

In [17]:
from scipy.stats import ttest_ind

def clinical_summary_table(df, numeric_cols):
    results = []

    for col in numeric_cols:
        group0 = df[df['is_stroke'] == 0][col].dropna()
        group1 = df[df['is_stroke'] == 1][col].dropna()

        mean0 = group0.mean()
        std0 = group0.std()

        mean1 = group1.mean()
        std1 = group1.std()

        # Independent t-test
        t_stat, p_value = ttest_ind(group0, group1, equal_var=False)

        results.append([
            col,
            f"{mean0:.2f} ± {std0:.2f}",
            f"{mean1:.2f} ± {std1:.2f}",
            round(p_value, 4)
        ])

    summary_df = pd.DataFrame(
        results,
        columns=["Variable", "Non-Stroke (Mean ± SD)", "Stroke (Mean ± SD)", "p-value"]
    )

    return summary_df.sort_values("p-value", ascending = False)


In [36]:
table = clinical_summary_table(df, numeric_cols=num_cols)

table.head(10)

/home1/bingi/.local/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,Variable,Non-Stroke (Mean ± SD),Stroke (Mean ± SD),p-value
74,max_diastolic_BP,111.98 ± 1116.20,109.85 ± 350.17,0.9098
72,max_Heart Rate,101.73 ± 21.96,102.31 ± 175.09,0.9068
22,mean_Anion Gap,14.49 ± 3.67,14.51 ± 2.63,0.7956
76,max_Spo2,101.24 ± 133.51,100.11 ± 25.31,0.5566
75,max_mean_BP,100.29 ± 108.29,209.56 ± 3503.03,0.2854
57,min_Mg,1.87 ± 0.33,1.88 ± 0.26,0.1864
18,Cocaine,0.00 ± 0.05,0.00 ± 0.03,0.1382
48,max_Platelet Count,223.70 ± 108.89,227.97 ± 86.96,0.1358
19,Sedative,0.00 ± 0.02,0.00 ± 0.00,0.0833
73,max_systolic_BP,150.94 ± 310.72,158.91 ± 23.68,0.0726


In [20]:
import pandas as pd
from scipy.stats import chi2_contingency

def categorical_summary_table(df, cat_cols):
    results = []

    for col in cat_cols:
        contingency = pd.crosstab(df[col], df['is_stroke'])

        # Perform Chi-square
        chi2, p, dof, expected = chi2_contingency(contingency)

        # Calculate percentages
        for category in contingency.index:
            non_stroke_total = contingency[0].sum()
            stroke_total = contingency[1].sum()

            non_stroke_count = contingency.loc[category, 0]
            stroke_count = contingency.loc[category, 1]

            non_stroke_pct = (non_stroke_count / non_stroke_total) * 100
            stroke_pct = (stroke_count / stroke_total) * 100

            results.append([
                f"{col} ({category})",
                f"{non_stroke_pct:.1f}%",
                f"{stroke_pct:.1f}%",
                round(p, 4)
            ])

    cat_df = pd.DataFrame(
        results,
        columns=["Variable", "Non-Stroke (%)", "Stroke (%)", "p-value"]
    )

    return cat_df.sort_values("p-value", ascending = False)


In [37]:
cat_table = categorical_summary_table(df, cat_cols)

cat_table.head(10)

,Variable,Non-Stroke (%),Stroke (%),p-value
0,first_careunit (Surgical Intensive Care Unit (...,100.0%,100.0%,1.0000
11,insurance (Medicare),52.4%,54.6%,0.0719
10,insurance (Medicaid),5.1%,3.7%,0.0719
12,insurance (Other),42.5%,41.8%,0.0719
21,gender (F),46.9%,51.2%,0.0067
22,gender (M),53.1%,48.8%,0.0067
1,last_careunit (Cardiac Vascular Intensive Care...,0.3%,0.0%,0.0000
6,last_careunit (Neuro Stepdown),0.2%,0.8%,0.0000
5,last_careunit (Neuro Intermediate),0.2%,2.4%,0.0000
4,last_careunit (Medical/Surgical Intensive Care...,0.7%,0.0%,0.0000


In [34]:

def train_test_summary_table(X_train, X_test, numeric_cols):
    results = []

    for col in numeric_cols:
        train_vals = X_train[col].dropna()
        test_vals = X_test[col].dropna()

        train_mean = train_vals.mean()
        train_std = train_vals.std()

        test_mean = test_vals.mean()
        test_std = test_vals.std()

        # Independent t-test
        t_stat, p_val = ttest_ind(train_vals, test_vals, equal_var=False)

        results.append([
            col,
            f"{train_mean:.2f} ± {train_std:.2f}",
            f"{test_mean:.2f} ± {test_std:.2f}",
            round(p_val, 6)
        ])

    df_result = pd.DataFrame(
        results,
        columns=[
            "Feature",
            "Training Set (Mean ± SD)",
            "Test Set (Mean ± SD)",
            "P-value"
        ]
    )

    return df_result.sort_values("P-value")


In [35]:
train_test_table = train_test_summary_table(df, test, num_cols)

train_test_table.head(10)

,Feature,Training Set (Mean ± SD),Test Set (Mean ± SD),P-value
0,los,3.89 ± 5.52,6.03 ± 8.32,0.0
14,Alcohol,0.03 ± 0.16,0.00 ± 0.00,0.0
15,Tobacco,0.03 ± 0.17,0.00 ± 0.00,0.0
10,sepsis,0.14 ± 0.34,0.22 ± 0.42,0.0
12,cancer,0.19 ± 0.39,0.37 ± 0.48,0.0
31,mean_Urea Nitrogen,24.43 ± 18.72,30.09 ± 22.32,0.0
24,mean_chloride,104.12 ± 5.49,105.42 ± 6.03,0.0
16,Opioid,0.00 ± 0.07,0.00 ± 0.00,0.0
61,min_Urea Nitrogen,23.14 ± 17.78,28.47 ± 21.21,0.0
46,max_Urea Nitrogen,25.71 ± 19.94,31.70 ± 23.63,0.0
